# Single all-insects model: curriculum vs naive vs classification

This notebook trains **one** YOLO-pose model on all insect groups at once, using
three strategies, and saves the results so they can be compared with
`compare_pose_results.py`.

- **Method 1 - curriculum (generalize then specialize).** The model first learns
  the keypoints common to every insect (body, thorax, legs, antennae, head),
  then progressively the hindwings, then the forewings. Each stage fine-tunes the
  previous stage's weights. The architecture keeps its 42 keypoint slots
  throughout; a stage simply masks the not-yet-learned keypoints to visibility 0.
- **Method 2 - naive.** A single training run on all groups, one class
  (`insect`), with all 42 keypoints active from the start.
- **Method 3 - naive with classification.** Same as method 2 but with **one
  class per insect group** instead of a single `insect` class. This adds a
  classification task to the multi-task loss; comparing it against method 2 shows
  whether learning to classify the group helps or hurts the pose estimation.

Every method produces a `results_<method>.xlsx` workbook (one summary row per
group) and per-group model copies `<method>__<group>.pt`, which is exactly what
the comparison script consumes.


## How to organize your datasets

All groups were already annotated with the **same 42 keypoints in the same
order** (parts an insect does not have are already at visibility 0), so no
remapping is needed. Concretely:

1. **Merge into one dataset** with a fixed layout. Labels are copied verbatim;
   only file names are prefixed by group to avoid collisions:

   ```
   datasets/all_insects/
     images/{train,val}/   group-prefixed images (beetles_img001.jpg, ...)
     labels/{train,val}/   the original 42-keypoint labels
     data.yaml             kpt_shape: [42, 3], names: {0: insect}, kpt_names: {...}
   ```
2. **Curriculum stages** reuse those images but with masked labels:

   ```
   datasets/stages/<stage>/
     images/{train,val}/   symlinks to the merged images
     labels/{train,val}/   only the stage's active keypoints are visible
     data.yaml
   ```

Your original per-group `data.yaml` files stay untouched: they are reused as-is
to evaluate the single model on each group separately, so the numbers line up
with the previous per-group experiments.


In [8]:
from pathlib import Path
import sys
import pandas as pd
from ultralytics import YOLO


sys.path.insert(1, '../train_eval_pose')

# --- Paths --------------------------------------------------------------
RESULTS_DIR = Path("./results/")          # shared with compare_pose_results.py
DATASETS_DIR = Path("../datasets")
MERGED_DIR  = Path("../datasets/all_insects")  # unified dataset (all groups)
STAGES_ROOT = Path("../datasets/stages")       # per-stage masked datasets
RUNS_DIR    = Path("./runs_single")           # Ultralytics run directory
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Original per-group datasets (unchanged, used for per-group evaluation).
GROUP_YAMLS = {
    "Coleoptera": "../datasets/Coleoptera/yolo-config.yaml",
    "Diptera": "../datasets/Diptera/yolo-config.yaml",
    "Hymenoptera": "../datasets/Hymenoptera/yolo-config.yaml",
    "Lepidoptera": "../datasets/Lepidoptera/yolo-config.yaml",
}

# --- Global 42-keypoint schema (this order IS the model output order) ---
KEYPOINT_NAMES = [
    "head-top", "head-left", "head-right", "left-eye", "right-eye", "neck",
    "thorax-left", "thorax-right", "thorax-bottom",
    "body-left", "body-right", "body-tip",
    "left-antenna-0", "left-antenna-1", "left-antenna-2",
    "right-antenna-0", "right-antenna-1", "right-antenna-2",
    "left-forewing-base", "left-forewing-tip", "left-forewing-front",
    "left-forewing-rear", "right-forewing-base", "right-forewing-tip",
    "right-forewing-front", "right-forewing-rear",
    "left-hindwing-base", "left-hindwing-tip", "left-hindwing-front",
    "left-hindwing-rear", "right-hindwing-base", "right-hindwing-tip",
    "right-hindwing-front", "right-hindwing-rear",
    "left-leg-0", "left-leg-1", "left-leg-2", "left-leg-3",
    "right-leg-0", "right-leg-1", "right-leg-2", "right-leg-3",
]
assert len(KEYPOINT_NAMES) == 42
GLOBAL_INDEX = {name: i for i, name in enumerate(KEYPOINT_NAMES)}


def keypoint_part(name):
    """Coarse body part used to define the curriculum stages."""
    if "forewing" in name:
        return "forewing"
    if "hindwing" in name:
        return "hindwing"
    return "general"


# --- Curriculum stages: general first, then hindwings, then forewings ---
STAGES = [
    {"name": "s1_general",   "active_parts": ["general"],                         "epochs": 90},
    {"name": "s2_hindwings", "active_parts": ["general", "hindwing"],             "epochs": 55},
    {"name": "s3_forewings", "active_parts": ["general", "hindwing", "forewing"], "epochs": 55},
]

# --- Shared training hyper-parameters -----------------------------------
BASE_MODEL   = "yolo26n-pose.pt"
IMGSZ        = 640
BATCH        = 16
LR0          = 0.01
POSE         = 12.0
KOBJ         = 1.0
FLIPLR       = 0.0    # keep 0.0 unless a correct flip_idx is set for 42 points
DEVICE       = "0"
NAIVE_EPOCHS = 90 + 55 + 55   # same total budget as the curriculum, for fairness

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


In [2]:
import shutil
import numpy as np
import yaml


def mask_label_line(values, active_set):
    """Zero out (visibility 0) every keypoint not in the active set."""
    box = values[:5]
    kpts = np.array(values[5:], float).reshape(42, 3)
    for k in range(42):
        if k not in active_set:
            kpts[k] = [0.0, 0.0, 0.0]
    return [*box, *kpts.reshape(-1)]


def iter_split_pairs(group_yaml, split):
    """Yield (image_path, label_path) for a group's split, standard YOLO layout."""
    data = yaml.safe_load(Path(group_yaml).read_text())
    root = DATASETS_DIR / data.get("path", Path(group_yaml).parent)
    #if not root.is_absolute():
     #   root = (Path(group_yaml).parent / root).resolve()
    field = Path(data.get(split, f"images/{split}"))
    img_dir = field if field.is_absolute() else (root / field)
    for img in sorted(img_dir.rglob("*")):
        if img.suffix.lower() in IMAGE_EXTS:
            label = Path(str(img).replace("/images/", "/labels/")).with_suffix(".txt")
            yield img, label


def link_or_copy(src, dst):
    if dst.exists():
        return
    try:
        dst.symlink_to(Path(src).resolve())
    except OSError:
        shutil.copy(src, dst)


In [3]:
def write_merged_yaml():
    content = {
        "path": str(MERGED_DIR.resolve()),
        "train": "images/train",
        "val": "images/val",
        "kpt_shape": [42, 3],
        "names": {0: "insect"},
        "kpt_names": {0: KEYPOINT_NAMES},
    }
    (MERGED_DIR / "data.yaml").write_text(yaml.safe_dump(content, sort_keys=False))


def build_merged_dataset():
    """Merge every group into a single dataset.

    All groups already share the same 42-keypoint schema (absent parts already
    annotated with visibility 0), so labels are copied verbatim; only the file
    names are prefixed by group to avoid collisions.
    """
    for split in ("train", "val"):
        (MERGED_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (MERGED_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

    for group, group_yaml in GROUP_YAMLS.items():
        for split in ("train", "val"):
            for img, label in iter_split_pairs(group_yaml, split):
                new_stem = f"{group}_{img.stem}"
                shutil.copy(img, MERGED_DIR / "images" / split / (new_stem + img.suffix))
                dst_label = MERGED_DIR / "labels" / split / (new_stem + ".txt")
                dst_label.write_text(label.read_text() if label.exists() else "")

    write_merged_yaml()
    print("Merged dataset written to", MERGED_DIR.resolve())


build_merged_dataset()


Merged dataset written to /home/tombellivier/Documents/CV/CV-for-GRIT/models/datasets/all_insects


In [4]:
def write_stage_yaml(stage_dir):
    content = {
        "path": str(stage_dir.resolve()),
        "train": "images/train",
        "val": "images/val",
        "kpt_shape": [42, 3],
        "names": {0: "insect"},
        "kpt_names": {0: KEYPOINT_NAMES},
    }
    (stage_dir / "data.yaml").write_text(yaml.safe_dump(content, sort_keys=False))


def build_stage_dataset(stage):
    """Create one stage: images symlinked, labels masked to active keypoints."""
    active = {i for i, n in enumerate(KEYPOINT_NAMES)
              if keypoint_part(n) in stage["active_parts"]}
    stage_dir = STAGES_ROOT / stage["name"]

    for split in ("train", "val"):
        (stage_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (stage_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
        for img in sorted((MERGED_DIR / "images" / split).glob("*")):
            link_or_copy(img, stage_dir / "images" / split / img.name)
        for label in sorted((MERGED_DIR / "labels" / split).glob("*.txt")):
            out_lines = []
            for line in label.read_text().splitlines():
                if not line.strip():
                    continue
                values = [float(x) for x in line.split()]
                out_lines.append(" ".join(
                    str(v) for v in mask_label_line(values, active)))
            (stage_dir / "labels" / split / label.name).write_text("\n".join(out_lines))

    write_stage_yaml(stage_dir)
    print(f"Stage '{stage['name']}' ready ({len(active)} active keypoints).")
    return stage_dir


STAGE_YAMLS = {s["name"]: build_stage_dataset(s) / "data.yaml" for s in STAGES}


Stage 's1_general' ready (26 active keypoints).
Stage 's2_hindwings' ready (34 active keypoints).
Stage 's3_forewings' ready (42 active keypoints).


## Method 1 - curriculum training

Each stage fine-tunes the previous stage's `best.pt` on a dataset where only the
stage's keypoints are visible. The per-stage learning curves are concatenated so
the comparison shows one line per stage.

In [ ]:

prev_weights = BASE_MODEL
staged_curves = []
epoch_offset = 0

for stage in STAGES:
    model = YOLO(prev_weights)
    model.train(
        data=str(STAGE_YAMLS[stage["name"]]),
        epochs=stage["epochs"], imgsz=IMGSZ, batch=BATCH,
        lr0=LR0, pose=POSE, kobj=KOBJ, fliplr=FLIPLR, device=DEVICE,
        project=str(RUNS_DIR), name=f"staged_{stage['name']}",
        exist_ok=True, verbose=False,
    )
    save_dir = Path(model.trainer.save_dir)
    staged_curves.append(load_curve(save_dir, stage["name"], epoch_offset))
    epoch_offset += stage["epochs"]
    prev_weights = str(save_dir / "weights" / "best.pt")

staged_best_weights = prev_weights
staged_curve_df = pd.concat(staged_curves, ignore_index=True)
print("Curriculum best weights:", staged_best_weights)


New https://pypi.org/project/ultralytics/8.4.87 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.80 🚀 Python-3.12.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7805MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../datasets/stages/s1_general/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=90, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-pose.pt, momentum=0.937, mosaic

## Method 2 - naive training

A single run on the merged dataset with all 42 keypoints active.

In [6]:
model = YOLO(BASE_MODEL)
model.train(
    data=str(MERGED_DIR / "data.yaml"),
    epochs=NAIVE_EPOCHS, imgsz=IMGSZ, batch=BATCH,
    lr0=LR0, pose=POSE, kobj=KOBJ, fliplr=FLIPLR, device=DEVICE,
    project=str(RUNS_DIR), name="naive_all", exist_ok=True, verbose=False,
)
naive_dir = Path(model.trainer.save_dir)
naive_best_weights = str(naive_dir / "weights" / "best.pt")
naive_curve_df = load_curve(naive_dir, "naive_all")
print("Naive best weights:", naive_best_weights)


New https://pypi.org/project/ultralytics/8.4.87 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.80 🚀 Python-3.12.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7805MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../datasets/all_insects/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-pose.pt, momentum=0.937, mosaic=1.0,

## Method 3 - naive training with classification

Same as method 2, but each insect group is its own class. This builds a parallel
dataset that reuses the merged images (symlinked) with labels whose class id is
the group index, and a `data.yaml` with one class name per group. Per-group
validation lists are written so the model can be evaluated group by group.

In [6]:
GROUP_TO_CLASS = {group: index for index, group in enumerate(GROUP_YAMLS)}
CLS_DIR = Path("datasets/all_insects_cls")


def group_of(stem):
    """Return the group a merged file belongs to, from its name prefix."""
    for group in GROUP_YAMLS:
        if stem.startswith(group + "_"):
            return group
    return None


def write_cls_yaml(path, val_field):
    content = {
        "path": str(CLS_DIR.resolve()),
        "train": "images/train",
        "val": val_field,
        "kpt_shape": [42, 3],
        "names": {index: group for group, index in GROUP_TO_CLASS.items()},
        "kpt_names": {index: KEYPOINT_NAMES for index in range(len(GROUP_TO_CLASS))},
    }
    Path(path).write_text(yaml.safe_dump(content, sort_keys=False))


def build_classification_dataset():
    """Reuse the merged images with per-group class ids in the labels."""
    for split in ("train", "val"):
        (CLS_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (CLS_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)
        for img in sorted((MERGED_DIR / "images" / split).glob("*")):
            link_or_copy(img, CLS_DIR / "images" / split / img.name)
        for label in sorted((MERGED_DIR / "labels" / split).glob("*.txt")):
            cls = GROUP_TO_CLASS.get(group_of(label.stem), 0)
            out_lines = []
            for line in label.read_text().splitlines():
                if not line.strip():
                    continue
                parts = line.split()
                parts[0] = str(cls)
                out_lines.append(" ".join(parts))
            (CLS_DIR / "labels" / split / label.name).write_text("\n".join(out_lines))

    write_cls_yaml(CLS_DIR / "data.yaml", "images/val")

    # Per-group validation lists so the model can be evaluated group by group.
    eval_yamls = {}
    for group in GROUP_YAMLS:
        images = sorted((CLS_DIR / "images" / "val").glob("*"))
        images = [p for p in images if group_of(p.stem) == group]
        list_path = CLS_DIR / f"{group}_val.txt"
        list_path.write_text("\n".join(str(p.resolve()) for p in images))
        group_yaml = CLS_DIR / f"data_cls_{group}.yaml"
        write_cls_yaml(group_yaml, str(list_path.resolve()))
        eval_yamls[group] = str(group_yaml)
    return str(CLS_DIR / "data.yaml"), eval_yamls


CLS_DATA_YAML, CLS_EVAL_YAMLS = build_classification_dataset()
print("Classification dataset ready:", CLS_DATA_YAML)


Classification dataset ready: datasets/all_insects_cls/data.yaml


In [12]:
model = YOLO(BASE_MODEL)
model.train(
    data=CLS_DATA_YAML,
    epochs=NAIVE_EPOCHS, imgsz=IMGSZ, batch=BATCH,
    lr0=LR0, pose=POSE, kobj=KOBJ, fliplr=FLIPLR, device=DEVICE,
    project=str(RUNS_DIR), name="naive_cls", exist_ok=True, verbose=False,
)
cls_dir = Path(model.trainer.save_dir)
cls_best_weights = str(cls_dir / "weights" / "best.pt")
cls_curve_df = load_curve(cls_dir, "naive_cls")
print("Classification best weights:", cls_best_weights)


New https://pypi.org/project/ultralytics/8.4.89 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.80 🚀 Python-3.12.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7805MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/all_insects_cls/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-pose.pt, momentum=0.937, mosaic=1.0

NameError: name 'load_curve' is not defined

## Evaluate every model per group and export

Each single model is evaluated on every group's original validation split, so the
workbook has one summary row per group, exactly like `train_eval_pose.py`. A copy
of the model is saved as `<method>__<group>.pt` so the inference-grid phase of the
comparison can render it on each group's example image.

In [ ]:
from datetime import datetime
from train_eval_pose import read_data_yaml, run_keypoint_pipeline

EVAL_CONF = 0.25
EVAL_IOU = 0.5


def evaluate_and_export(model_weights, run_tag, curve_df, group_eval_yamls=None):
    group_eval_yamls = group_eval_yamls or GROUP_YAMLS
    model = YOLO(str(model_weights))
    summary_rows, per_keypoint_frames = [], []

    for group, group_yaml in group_eval_yamls.items():
        info = read_data_yaml(group_yaml)
        metrics = model.val(data=group_yaml, imgsz=IMGSZ, device=DEVICE, verbose=False)
        accumulator, n_images = run_keypoint_pipeline(model, group_yaml, info, EVAL_CONF, EVAL_IOU)

        row = {
            "group": group, "num_val_images": n_images,
            "pose_map": float(metrics.pose.map),
            "pose_map50": float(metrics.pose.map50),
            "pose_map75": float(metrics.pose.map75),
            "box_map": float(metrics.box.map),
            "box_map50": float(metrics.box.map50),
        }
        row.update(accumulator.summary())
        summary_rows.append(row)

        per_keypoint = accumulator.per_keypoint_frame(info["kpt_names"])
        per_keypoint.insert(0, "group", group)
        per_keypoint_frames.append(per_keypoint)

        # Per-group copy so the comparison's inference grid finds one .pt per group.
        shutil.copy(model_weights, RESULTS_DIR / f"{run_tag}__{group}.pt")

    summary_df = pd.DataFrame(summary_rows)
    front = ["group"] + [c for c in summary_df.columns if c != "group"]
    summary_df = summary_df[front]
    per_keypoint_df = pd.concat(per_keypoint_frames, ignore_index=True)

    metadata = {
        "run_tag": run_tag, "model": BASE_MODEL, "training": run_tag,
        "imgsz": IMGSZ, "batch": BATCH, "lr0": LR0, "pose": POSE, "kobj": KOBJ,
        "fliplr": FLIPLR, "device": str(DEVICE),
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    }
    metadata_df = pd.DataFrame(
        {"field": list(metadata.keys()), "value": list(metadata.values())})

    out_path = RESULTS_DIR / f"results_{run_tag}.xlsx"
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        metadata_df.to_excel(writer, sheet_name="metadata", index=False)
        summary_df.to_excel(writer, sheet_name="summary", index=False)
        per_keypoint_df.to_excel(writer, sheet_name="per_keypoint", index=False)
        curve_df.to_excel(writer, sheet_name="learning_curves", index=False)
    print("Wrote", out_path)


6(staged_best_weights, "single_staged", staged_curve_df)
evaluate_and_export(naive_best_weights, "single_naive", naive_curve_df)
evaluate_and_export(cls_best_weights, "single_naive_cls", cls_curve_df,
                    group_eval_yamls=CLS_EVAL_YAMLS)


Ultralytics 8.4.80 🚀 Python-3.12.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7805MiB)
YOLO26n-pose summary (fused): 132 layers, 4,493,469 parameters, 0 gradients, 15.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 8769.2±3278.9 MB/s, size: 258.0 KB)
val: Scanning /home/tombellivier/Documents/CV/CV-for-GRIT/models/datasets/Coleoptera/labels/val.cache... 83 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 83/83 69.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 5.7it/s 1.0s0.5s
                   all         83         83          1      0.997      0.995       0.91      0.976      0.973      0.971      0.484
Speed: 2.3ms preprocess, 3.4ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /home/tombellivier/Documents/CV/CV-for-GRIT/models/training/runs/pose/val-11
Ultralytics 8.4.80 🚀 Python-3.12.3 torch-2.11.0+cu130 CUDA:0 (NVID

## Compare the three methods

Reuse the existing comparison script. It places `single_staged`, `single_naive`
and `single_naive_cls` side by side: group comparisons, learning curves (one line
per stage for the curriculum), confidence-vs-error grids, the mAP50 / PCK
heatmaps, and the qualitative inference grids on the example images.

In [ ]:
import subprocess, sys

# subprocess.run([
#     sys.executable, "compare_pose_results.py",
#     "--results-dir", str(RESULTS_DIR),
#     "--out-dir", "comparison",
# ])


### Reading the outputs

- `comparison/comparison_summary.xlsx` - `all_runs` (three methods x 4 groups),
  `best_per_group`, and the per-metric pivots.
- `comparison/figures/` - `heatmap__pose_map50.png`, `heatmap__pck_0.1.png`,
  the per-method learning curves and confidence-vs-error grids, and one
  `inference__single_*.png` per method.

Two questions are answered by the heatmaps (rows = the three methods, columns =
datasets) and by `best_per_group`: does generalize-then-specialize beat naive
training, and does adding the group-classification task (`single_naive_cls` vs
`single_naive`) help the pose - and if so, for which groups.
